# Edge-ML Data Processing and Feature Engineering
This notebook loads datasets from Edge-ML, filters and preprocesses accelerator data (including trimming transitional pocket movements), applies rolling-window aggregations to extract statistical features, and visualizes the structured metrics across multiple axes.

### 1. Import Libraries and Load Environment Variables

In [3]:
%pip install edge-ml dotenv matplotlib seaborn numpy pandas scikit-learn optuna skl2onnx onnx torch lightgbm

Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip show edge-ml

Name: edge-ml
Version: 0.4.0
Summary: ```python
Home-page: 
Author: 
Author-email: KIT/TECO <info@edge-ml.org>
License: 
Location: /home/till/kit-uni/ContSys/venv/.venv/lib/python3.10/site-packages
Requires: 
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [5]:
import os 
from dotenv import load_dotenv
from edgeml import Dataset, DatasetReceiver
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# # Get API Keys
load_dotenv()
write_key = os.getenv("EDGE_WRITE_KEY")
read_key = os.getenv("EDGE_READ_KEY")

### 2. Initialize Edge-ML Receiver

In [ ]:
# Uses Read- and Write-Key!
receiver = DatasetReceiver(
    backendURL="https://app.edge-ml.org",
    readKey="d509fe191bbaa26ff61b4ecdfda7c2df",
    writeKey="6b7f5daede0b7b1dc72b30f859a43864"
)

receiver25 = DatasetReceiver(
  backendURL='https://beta.edge-ml.org',
  readKey='02a4fad735d3308b68672ddb7593f047',
  writeKey='5fe6e50c3fb5001531bbd8e03a8c591f'
)


### 3. Helper Functions for Dataset Discovery and Querying

In [7]:
# Gather all Datasets that have a certain name
def gather(datasets: list[Dataset], name: str):
    return [ds for ds in datasets if name.lower().startswith(getName(ds))]

def getParticipant(ds: Dataset):
    participant = ds.metaData.get("participantId") or ds.metaData.get("participant")
    return participant if participant is not None else None
    
def getWithParticipant(datasets: list[Dataset]): # type: ignore
    return [ds for ds in datasets if getParticipant(ds) != None]

# Gather own Datasets that have a certain name
def gatherOwn(datasets: list[Dataset], name: str):
    return [ds for ds in getOwn(datasets=datasets) if getName(ds) == name]    

# Get the label of the dataset (e.g. "swimming")
def getName(ds: Dataset):
    if ds.labelings == [] or ds.labelings == None:
        return "!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!"
    return ds.labelings[0].labels[-1].name.lower()

# Get the dataset that I have created
def getOwn(datasets: list[Dataset]):
    return [ds for ds in datasets if ds.name.startswith("uoihp")]    

### 4. Fetch Label-Specific Datasets

In [8]:
datasets26 = receiver.datasets
datasets25 = receiver25.datasets

print(len(datasets25))
print(len(datasets26))

datasets = datasets26 + datasets25 

datasets_with_participant = getWithParticipant(datasets=datasets) 
print(f"Number of datasets: {len(datasets)}")
print(f"Number of datasets with pariticapant metadata: {len(datasets_with_participant)}")

# sitting
sitting_ds = gather(datasets_with_participant, "sitting")
print(f"Number of datasets with sitting: {len(sitting_ds)}")

walking_ds = gather(datasets_with_participant, "walking")
print(f"Number of datasets with walking: {len(walking_ds)}")

running_ds = gather(datasets_with_participant, "running")
print(f"Number of datasets with running: {len(running_ds)}")

stairs_ds  = gather(datasets_with_participant, "stairs")
print(f"Number of datasets with stairs: {len(stairs_ds)}")

standing_ds  = gather(datasets_with_participant, "standing")
print(f"Number of datasets with standing: {len(standing_ds)}")

participant_label = {}

def mapParticipant(datasets: list[Dataset], participant_label: dict[str, int]):
  for ds in datasets:
    participant = getParticipant(ds)
    if participant not in participant_label:
      participant_label[participant] = len(participant_label)

mapParticipant(sitting_ds, participant_label)
mapParticipant(walking_ds, participant_label)
mapParticipant(running_ds, participant_label)
mapParticipant(stairs_ds, participant_label)
mapParticipant(standing_ds, participant_label)

print(participant_label)

629
129
Number of datasets: 758
Number of datasets with pariticapant metadata: 437
Number of datasets with sitting: 87
Number of datasets with walking: 95
Number of datasets with running: 19
Number of datasets with stairs: 13
Number of datasets with standing: 42
{'Till Hoffmann': 0, 'ukjvp_s1': 1, 'ukjvp_s2': 2, '01': 3, 'testify': 4, 'P001': 5, 'ukjvp_s3': 6, 'ukjvp_s4': 7, 'gh': 8, 'zh': 9, 'jh': 10, 'lm': 11, '10d83': 12, '11416': 13, '1a045': 14, '1a3c3': 15, '15b85': 16, '191d4': 17, '1f683': 18, '17dcc': 19, '18582': 20, '1dd54': 21, '11cc1': 22, '16742': 23, '1b728': 24, 'uspgb': 25, '12f77': 26, 'uupiw': 27, '1e60e': 28, 'vand': 29, 'Vand': 30, '1ea25': 31, '1942f': 32, 'gilbert': 33, '1fcb8': 34, '11692': 35, '19205': 36, 'ukjvp_w1': 37, 'ukjvp_w2': 38, 'ukjvp_w3': 39, 'ukjvp_w4': 40, 'ukjvp_w5': 41, 'ukjvp_w7': 42, 'ukjvp_w8': 43, 'ukjvp_w9': 44, 'ukjvp_w10': 45, 'ukjvp_walkin_wm': 46, '19ee7': 47, '12358': 48, '1254d': 49, '1b949': 50, '150c9': 51, '1c07d': 52, '1fa30': 53, 

### 5. Data Transformation and Edge Trimming Functions

In [9]:
# Resample data because of different sample rates
# Convert datasets into one dataframe per activity
import numpy as np
import pandas as pd

def datasets_to_df_with_trimmed_edges(datasets: list[Dataset], activity: str, participant_label):
    frames = []

    for ds in datasets:
        # Create a copy so we don't accidentally modify the original variable
        ds.loadData()
        df = pd.DataFrame(ds.data)


        try:
            # Find which set of columns actually exists in this dataset
            if all(col in df.columns for col in ["accX", "accY", "accZ"]):
                acc_cols = ["accX", "accY", "accZ"]
            elif all(col in df.columns for col in ["acceleration.x", "acceleration.y", "acceleration.z"]):
                acc_cols = ["acceleration.x", "acceleration.y", "acceleration.z"]
            elif all(col in df.columns for col in ["sensor_acc_X", "sensor_acc_Y", "sensor_acc_Z"]):
                acc_cols = ["acc_X", "acc_Y", "acc_Z"]
            else:
                raise KeyError(f"Could not find a matching accelerometer column format in this dataset.\n DataFrame: {df}")

            # Ensure the columns match your required features
            df = df[["time"] + acc_cols]

            df = df.rename(columns={
                "sensor_accX": "accX",
                "sensor_accY": "accY",
                "sensor_accZ": "accZ"
            })

            df = df.rename(columns={
                "acceleration.x": "accX",
                "acceleration.y": "accY",
                "acceleration.z": "accZ"
            })

            df = df.rename(columns={
                "acc_X": "accX",
                "acc_Y": "accY",
                "acc_Z": "accZ"
            })

            # add new feature
            # 3D acceleration amplitude
            # ---------------------------------- 
            df["accM"] = np.sqrt(
                df["accX"]**2 + 
                df["accY"]**2 + 
                df["accZ"]**2
            )
            # ---------------------------------- 


            df["time"] = pd.to_datetime(df["time"], unit="ms")
            
            df = df.sort_values("time")        
            start_time = df["time"].min() + pd.Timedelta(seconds=2)
            end_time = df["time"].max() - pd.Timedelta(seconds=2)
            
            # Keep only the data in the middle
            df = df[(df["time"] >= start_time) & (df["time"] <= end_time)]

            df["activity"] = activity
            df["subject"]  = participant_label[getParticipant(ds)]
            frames.append(df)
        
        finally:
            continue

    return pd.concat(frames, ignore_index=True)

In [10]:
sitting_df = datasets_to_df_with_trimmed_edges(sitting_ds, "sitting", participant_label).dropna()
walking_df = datasets_to_df_with_trimmed_edges(walking_ds, "walking", participant_label).dropna()
running_df = datasets_to_df_with_trimmed_edges(running_ds, "running", participant_label).dropna()
stairs_df = datasets_to_df_with_trimmed_edges(stairs_ds, "stairs",  participant_label).dropna()
standing_df = datasets_to_df_with_trimmed_edges(standing_ds, "standing",  participant_label).dropna()

all_df = [sitting_df, walking_df, running_df, stairs_df, standing_df]

df = pd.concat(all_df, ignore_index=True)

### 7. Feature Extraction Using Rolling Windows

In [11]:
# 1. Set the time as the index and sort it chronologically
df_sorted = df.set_index("time").sort_index()

# 2. Group by activity, apply the rolling window, and aggregate
df_windowed = (
    df_sorted.groupby(["activity", "subject"])
      .rolling(window="1s")  # 'on="time"' is no longer needed since it's the index
      .agg(["mean", "std"])
      .dropna()
      .reset_index()          # Brings 'activity' and 'time' back as normal columns
)

# 3. Clean up MultiIndex column names (e.g., 'accX_mean')
df_windowed.columns = [
    f"{col[0]}_{col[1]}" if isinstance(col, tuple) and col[1] else col[0]
    for col in df_windowed.columns
 ]


df_windowed

,activity,subject,time,accX_mean,accX_std,accY_mean,accY_std,accZ_mean,accZ_std,accM_mean,accM_std
0,running,11,2026-07-04 17:31:45.959,3.150000,1.343503,12.600000,6.646803,0.300000,5.939697,13.710298,6.547172
1,running,11,2026-07-04 17:31:45.985,2.900000,1.044031,9.500000,7.135825,-1.133333,4.878866,11.044871,6.538060
2,running,11,2026-07-04 17:31:46.011,2.850000,0.858293,7.500000,7.067295,-1.525000,4.059865,9.309263,6.367639
3,running,11,2026-07-04 17:31:46.039,2.640000,0.879204,6.080000,6.895070,-1.620000,3.522357,7.991469,6.252441
4,running,11,2026-07-04 17:31:46.065,2.316667,1.116094,4.950000,6.759807,-1.366667,3.211023,6.825389,6.279560
...,...,...,...,...,...,...,...,...,...,...,...
246293,walking,53,2025-09-14 21:09:27.670,0.226333,0.985674,0.254333,1.034328,-0.150667,2.607103,2.692978,1.266605
246294,walking,53,2025-09-14 21:09:27.686,0.188500,1.028927,0.258000,1.042636,-0.161667,2.602051,2.703578,1.270290
246295,walking,53,2025-09-14 21:09:27.704,0.157667,1.049985,0.274667,1.082258,-0.182167,2.587695,2.711971,1.275233
246296,walking,53,2025-09-14 21:09:27.720,0.124000,1.052943,0.282000,1.097529,-0.190667,2.581369,2.712665,1.275512


### 8. Prepare Data for Multi-axis Visualization

In [12]:
# 1. Melt the dataframe to long-form
df_melted = df_windowed.melt(
    id_vars=["activity", "time", "subject"],
    value_vars=[
        "accX_mean", "accX_std",
        "accY_mean", "accY_std",
        "accZ_mean", "accZ_std",
        "accM_mean", "accM_std"
    ],
    var_name="Sensor_Metric",
    value_name="Value"
)

# 2. Split the names into separate 'Axis' and 'Metric' columns
df_melted["Axis"] = df_melted["Sensor_Metric"].apply(lambda x: x.split("_")[0].replace("acc", ""))
df_melted["Metric"] = df_melted["Sensor_Metric"].apply(lambda x: x.split("_")[1])

# Check right now if 'Metric' actually exists and has data
print(df_melted.head())


  activity                    time  subject Sensor_Metric     Value Axis  \
0  running 2026-07-04 17:31:45.959       11     accX_mean  3.150000    X   
1  running 2026-07-04 17:31:45.985       11     accX_mean  2.900000    X   
2  running 2026-07-04 17:31:46.011       11     accX_mean  2.850000    X   
3  running 2026-07-04 17:31:46.039       11     accX_mean  2.640000    X   
4  running 2026-07-04 17:31:46.065       11     accX_mean  2.316667    X   

  Metric  
0   mean  
1   mean  
2   mean  
3   mean  
4   mean  


### 10. Train Models

In [14]:
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.model_selection import LeaveOneGroupOut, KFold, cross_val_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier

feature_cols = [
    "accX_mean", "accY_mean", "accZ_mean", "accM_mean",
    "accX_std", "accY_std", "accZ_std", "accM_std"
]

# SMALL DATA BECAUSE BIG DATA TAKES TOO LONG RIGHT NOW
# df_windowed = df_small

X = df_windowed[feature_cols].values

groups = df_windowed["subject"].values  # Tracking array for subjects

# Encode our text labels ('walking', 'sitting'...) to 0, 1, 2
le = LabelEncoder()
y = le.fit_transform(df_windowed["activity"])

In [82]:
%pip install kornia

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 2.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 3.2 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [92]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

import kornia.losses as losses 

# ----------------------------
# Model
# ----------------------------

class ActivityNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(8, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(32, 5)
        )

    def forward(self, x):
        return self.net(x)



# ----------------------------
# Training function
# ----------------------------

def train_model(X_train, y_train, epochs=50):

    model = ActivityNet()

    dataset = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long)
    )

    loader = DataLoader(
        dataset,
        batch_size=64,
        shuffle=True
    )


    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001,
        # add weight decay
        weight_decay=1e-4
    )


    # criterion = losses.FocalLoss(alpha=0.25, gamma=2.0, reduction="mean")

    criterion = nn.CrossEntropyLoss()

    model.train()

    for epoch in range(epochs):

        for xb, yb in loader:

            optimizer.zero_grad()

            output = model(xb)

            loss = criterion(output, yb)

            loss.backward()

            optimizer.step()


    return model



# ----------------------------
# LOGO evaluation
# ----------------------------

n_splits = 3
gkv = GroupKFold(n_splits=n_splits)

accuracies = []
f1_scores = []

all_predictions = []
all_labels = []


fold = 1


for train_idx, test_idx in gkv.split(
        X,
        y,
        groups
):

    print(f"\nFold {fold}/{n_splits}")
    

    X_train = X[train_idx]
    X_test  = X[test_idx]

    y_train = y[train_idx]
    y_test  = y[test_idx]


    # ------------------------
    # Normalize ONLY using training participants
    # ------------------------

    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)

    X_test = scaler.transform(X_test)

    # ------------------------
    # Train
    # ------------------------

    model = train_model(
        X_train,
        y_train
    )


    # ------------------------
    # Test participant
    # ------------------------

    model.eval()

    with torch.no_grad():

        logits = model(
            torch.tensor(
                X_test,
                dtype=torch.float32
            )
        )

        predictions = torch.argmax(
            logits,
            dim=1
        ).numpy()



    acc = accuracy_score(
        y_test,
        predictions
    )

    f1 = f1_score(
        y_test,
        predictions,
        average="weighted"
    )


    accuracies.append(acc)
    f1_scores.append(f1)


    all_predictions.extend(predictions)
    all_labels.extend(y_test)


    print(
        f"Participant accuracy: {acc:.3f}, "
        f"F1: {f1:.3f}"
    )


    fold += 1



# ----------------------------
# Final results
# ----------------------------

print("\n===================")

print(
    f"LOGO Accuracy: "
    f"{np.mean(accuracies):.3f} "
    f"+/- {np.std(accuracies):.3f}"
)

print(
    f"LOGO F1: "
    f"{np.mean(f1_scores):.3f} "
    f"+/- {np.std(f1_scores):.3f}"
)


print("\nConfusion matrix:")

print(
    confusion_matrix(
        all_labels,
        all_predictions
    )
)




Fold 1/3
Participant accuracy: 0.817, F1: 0.809

Fold 2/3
Participant accuracy: 0.714, F1: 0.708

Fold 3/3
Participant accuracy: 0.626, F1: 0.582

LOGO Accuracy: 0.719 +/- 0.078
LOGO F1: 0.700 +/- 0.093

Confusion matrix:
[[14332  2929    32  1053  2394]
 [ 8949 69694  1869  2672  7145]
 [  930   920   427    31  9498]
 [ 3646  8612    54  4606  3184]
 [ 2977  6957  1319  4070 87998]]


In [80]:
print(le.classes_)

['running' 'sitting' 'stairs' 'standing' 'walking']


### 14. Bayesian Optimization

In [70]:
import optuna
from optuna.samplers import RandomSampler, GPSampler, GridSampler, QMCSampler, BruteForceSampler
import warnings
import optuna
from sklearn.model_selection import GroupKFold
import lightgbm as lgb

optuna.logging.set_verbosity(optuna.logging.ERROR)

warnings.filterwarnings("ignore")

X = np.asarray(X, dtype=np.float32)
y = np.asarray(y)
groups = np.asarray(groups)

gkf = GroupKFold(n_splits=3)
splits = list(gkf.split(X, y, groups))

def objective(trial):
  n_estimators = trial.suggest_int("n_estimators", 50, 100)
  max_depth = trial.suggest_int("max_depth", 3, 6)
  max_features = trial.suggest_int("max_features", 3, 8)

  clf = RandomForestClassifier(
    max_depth=max_depth,
    n_estimators=n_estimators,
    max_features=max_features,
    random_state=42,
    n_jobs=-1
  )
  scores = []

  for train_idx, test_idx in splits:
      clf.fit(
          X[train_idx],
          y[train_idx]
      )

      pred = clf.predict(X[test_idx])

      scores.append(
          f1_score(
              y[test_idx],
              pred,
              average="macro"
          )
      )
  
  print("Score:", np.mean(scores))

  return np.mean(scores)

In [71]:
gp  = GPSampler(seed=42)

sampler = gp
study = optuna.create_study(sampler=sampler, direction="maximize")
study.optimize(objective, n_trials=50)

best_trial = study.best_trial
print("Best value: ", best_trial.value)
print("Parameters that achieve the best value: ", best_trial.params)

Score: 0.46640323577715065
Score: 0.48179375385461043
Score: 0.4782736362387398


KeyboardInterrupt: 

In [17]:
%pip install nbformat

Note: you may need to restart the kernel to use updated packages.


In [66]:
import optuna.visualization as vis

fig = vis.plot_optimization_history(study)
fig.show()

fig = vis.plot_param_importances(study)
fig.show()

fig = vis.plot_parallel_coordinate(study)
fig.show()

fig = vis.plot_slice(study)
fig.show()

fig = vis.plot_contour(study)
fig.show()

fig = vis.plot_edf(study)
fig.show()

fig = vis.plot_rank(study)
fig.show()

fig = vis.plot_timeline(study)
fig.show()

fig = vis.plot_intermediate_values(study) 
fig.show()

In [72]:
print(X.shape[1])

print(X.columns)

8


AttributeError: 'numpy.ndarray' object has no attribute 'columns'

In [20]:
from skl2onnx import convert_sklearn
import torch
import torch.nn as nn
import onnx
from skl2onnx.common.data_types import FloatTensorType

TARGET_OPSET = 17

# Parameters that achieve the best value:  {'n_estimators': 31, 'max_depth': 6, 'max_features': 1}
rf = RandomForestClassifier(
  n_estimators=31,
  max_depth=6,
  max_features=1,
  random_state=42
)

rf.fit(X, y)

num_features = X.shape[1]
initial_type = [('extracted_features', FloatTensorType([None, num_features]))]

rf_onnx = convert_sklearn(
    rf,
    initial_types=initial_type,
    target_opset=TARGET_OPSET
)

# 1. Define PyTorch Math Preprocessor
class SignalPreprocessingGraph(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        # x shape: (1, 3, N) -> [accX, accY, accZ] over time
        accX = x[:, 0, :]
        accY = x[:, 1, :]
        accZ = x[:, 2, :]

        accM = torch.sqrt(accX**2 + accY**2 + accZ**2)

        accX_mean = torch.mean(accX, dim=1, keepdim=True)
        accY_mean = torch.mean(accY, dim=1, keepdim=True)
        accZ_mean = torch.mean(accZ, dim=1, keepdim=True)
        accM_mean = torch.mean(accM, dim=1, keepdim=True)

        accX_std = torch.std(accX, dim=1, keepdim=True, correction=0)
        accY_std = torch.std(accY, dim=1, keepdim=True, correction=0)
        accZ_std = torch.std(accZ, dim=1, keepdim=True, correction=0)
        accM_std = torch.std(accM, dim=1, keepdim=True, correction=0)

        features = torch.cat([
            accX_mean, accY_mean, accZ_mean, accM_mean,
            accX_std,  accY_std,  accZ_std,  accM_std
        ], dim=1)
        
        return features

prep_graph = SignalPreprocessingGraph().eval()
dummy_sensor_input = torch.randn(1, 3, 50)

torch.onnx.export(
    prep_graph,
    dummy_sensor_input,
    "prep_graph.onnx",
    input_names=["raw_sensor_data"],
    output_names=["extracted_features"],
    dynamic_axes={"raw_sensor_data": {2: "num_samples"}},
    opset_version=TARGET_OPSET,
    dynamo=False   # important
)

prep_onnx_model = onnx.load("prep_graph.onnx")
rf_onnx_model = onnx.load_from_string(rf_onnx.SerializeToString())

# Make IR versions identical
rf_onnx_model.ir_version = prep_onnx_model.ir_version

# Remove the opset conflict
# Keep the RF tree model's opsets
prep_onnx_model.opset_import.clear()

for opset in rf_onnx_model.opset_import:
    prep_onnx_model.opset_import.append(opset)

baked_model = onnx.compose.merge_models(
    prep_onnx_model,
    rf_onnx_model,
    io_map=[
        ("extracted_features", "extracted_features")
    ],
)

onnx.checker.check_model(baked_model)

onnx.save(
    baked_model,
    "sensor_classifier.onnx"
)

print("Successfully saved unified pipeline")

Successfully saved unified pipeline


In [21]:
import onnx

model = onnx.load("sensor_classifier.onnx")

for inp in model.graph.input:
    print(inp.name, inp.type.tensor_type.shape)

for out in model.graph.output:
    print("OUTPUT:", out.name)

raw_sensor_data dim {
  dim_value: 1
}
dim {
  dim_value: 3
}
dim {
  dim_param: "num_samples"
}

OUTPUT: output_label
OUTPUT: output_probability


In [22]:
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession("sensor_classifier.onnx")

x = np.random.randn(1, 3, 149).astype(np.float32)

outputs = session.run(
    None,
    {"raw_sensor_data": x}
)

print(outputs)

[array([4], dtype=int64), [{0: 0.01900552026927471, 1: 0.058917272835969925, 2: 0.0128988828510046, 3: 0.035935964435338974, 4: 0.873242199420929}]]
